In [1]:
import pandas as pd
import numpy as np
import swifter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import string


c:\Users\Bangkit\anaconda3\envs\QF634\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv('Data/news_data_final_train_labelled.csv')
test = pd.read_csv('Data/news_data_final_test_labelled.csv')

In [3]:
train

,date,title,source,topic,sentiment_label,Ticker
0,2014-01-06,Apple Inc. Makes Diversity Changes in Bylaws,San Francisco Business Times,"Technology Hardware, Storage and Peripherals",1.0,AAPL
1,2014-01-10,Apple Solicits Proxies from Shareholders Again...,Other,"Technology Hardware, Storage and Peripherals",1.0,AAPL
2,2014-01-13,Apple Wins Appeal against Google's Motorola,Other,"Technology Hardware, Storage and Peripherals",1.0,AAPL
3,2014-01-14,Rembrandt Patent Innovations and Rembrandt Sec...,Business Wire,"Technology Hardware, Storage and Peripherals",1.0,AAPL
4,2014-01-15,VirnetX Holding Corp. Adds Claims in Apple Inc...,Other,"Technology Hardware, Storage and Peripherals",-1.0,AAPL
...,...,...,...,...,...,...
500564,2023-12-07,FinTech: Samsung Electronics Ties With Masterc...,https://dailyalts.com/samsung-electronics-ties...,finance,-1.0,FMC
500565,2023-12-07,Digital Assets: Robinhood Debuts Crypto Tradin...,https://dailyalts.com/robinhood-debuts-crypto-...,finance,-1.0,FMC
500566,2023-12-07,Artificial Intelligence: AMD Takes On Rivals I...,https://dailyalts.com/amd-takes-on-rivals-in-t...,finance,-1.0,FMC
500567,2023-12-15,FMC Corporation Declares Regular Quarterly Div...,PR Newswire,Fertilizers and Agricultural Chemicals,1.0,FMC


In [4]:

# Download NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')


def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
# Preprocessing function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize text
    tokens = word_tokenize(text.lower())
    
    # Get POS tags
    pos_tags = pos_tag(tokens)

    # Remove punctuation and stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags 
              if word not in stop_words and word not in string.punctuation]
    return ' '.join(tokens)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [ ]:

train['processed_title'] = train['title'].swifter.apply(preprocess_text)
# val['processed_title'] = val['title'].swifter.apply(preprocess_text)
test['processed_title'] = test['title'].swifter.apply(preprocess_text)


# Prepare data for Naive Bayes
X_train = train['processed_title']
y_train  = train['sentiment_label']

# X_val = val['processed_title']
# y_val  = val['sentiment_label']

X_test = test['processed_title']
y_test  = test['sentiment_label']

# Convert text to numerical features using CountVectorizer
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Train a Naive Bayes classifier
nb = MultinomialNB()
nb.fit(X_train_vectorized, y_train)

# Make predictions
y_pred = nb.predict(X_test_vectorized)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
